In [ ]:
from src.data_loader import load_raw_data, get_evaluation_dataset
import pandas as pd
from litellm import completion
from pydantic import BaseModel, Field

In [2]:
df_examples, df_products = load_raw_data('../data/raw')

In [3]:
x = get_evaluation_dataset(df_examples, df_products)

In [ ]:
subset = pd.read_csv('../data/processed/subset_to_review.csv')

In [20]:
class MyResponse(BaseModel):
    acceptable: bool = Field(description="True if the product is an exact match for the query. False otherwise")
    reasoning: str = Field(description="Explain why the query and the product information match")

In [ ]:
import os
import json


In [21]:
system_prompt = '''
You are an expert auditing labeled data with user search queries and returned product matches. 
Your goal is to tell us if product and query information matches or not. 
'''

In [32]:
for idx, row in subset.iterrows():
    user_prompt = f'''Query:{subset.iloc[0].query}\nProduct Information:\n\tTitle: {subset.iloc[0].product_title}\n\tDescription:{subset.iloc[0].product_description}\n\tBrand: {subset.iloc[0].product_brand}\n\tBullets: {subset.iloc[0].product_bullet_point}
    '''
    response = completion(
        model='gemini/gemini-2.5-flash',
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=MyResponse
    )
    solution = MyResponse.model_validate_json(response['choices'][0]['message'].content)
    break
